In [1]:
import subprocess
import os
import pandas as pd
import numpy as np
from core.reader import read_ansys_csv, load_experimental_data

ANSYS_EXE_PATH = r"D:\Program Files\ANSYS Inc\ANSYS Student\v252\ansys\bin\winx64\MAPDL.exe" 
WORKING_DIR = os.getcwd()

def run_ansys_simulation(params):
    with open('chab_params.txt', 'w') as f:
        for p in params:
            f.write(f"{p}\n")

    input_file = "chab.mac"
    output_file = "ansys.out"
    
    cmd = [
        ANSYS_EXE_PATH, 
        "-b",
        "-j", "opt_run",
        "-dir", WORKING_DIR, 
        "-i", input_file, 
        "-o", output_file
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        print("Ошибка ANSYS:", e)
        return None

    try:
        df_res = read_ansys_csv("chab.csv")
        return df_res
    except Exception as e:
        print(f"Ошибка чтения CSV: {e}")
        return None

In [2]:
real_experiment_data_folder = "."
df_exp = load_experimental_data(real_experiment_data_folder)

zero_row = pd.DataFrame(0.0, columns=df_exp.columns, index=[0])
zero_row['Time'] = 1.0
df_exp = pd.concat([zero_row, df_exp]).reset_index(drop=True)

def objective_function(params):
    """
    Считает ошибку между экспериментом и моделью Шабоша.
    params: [sig_y, c1, g1, c2, g2, c3, g3]
    """
    print(f"Simulating: {params}")
    
    df_ansys = run_ansys_simulation(params)
    
    if df_ansys is None or len(df_ansys) != len(df_exp):
        return 1e9

    mse_zz = np.mean((df_ansys['S_ZZ'] - df_exp['S_ZZ'])**2)
    mse_tt = np.mean((df_ansys['S_TT'] - df_exp['S_TT'])**2)
    mse_tz = np.mean((df_ansys['S_TZ'] - df_exp['S_TZ'])**2)
    
    total_error = mse_zz + mse_tt + mse_tz
    print(f"Error: {total_error:.2f}")
    return total_error

In [3]:
import psutil

def kill_ansys_processes():
    for proc in psutil.process_iter():
        if proc.name() in ['ANSYS.exe', 'MAPDL.exe', 'ansys.exe']:
            proc.kill()

kill_ansys_processes()

In [4]:
from scipy.optimize import differential_evolution

# Границы поиска для каждого параметра
# [sig_y, c1, g1, c2, g2, c3, g3]
bounds = [
    (200, 400),      # Sig_Y
    (1e4, 5e5),      # C1 (Жесткая кинематика)
    (100, 5000),     # gamma1 (Быстрое насыщение)
    (1e3, 5e4),      # C2
    (10, 500),       # gamma2
    (100, 1e4),      # C3
    (0, 100)         # gamma3 (Линейная часть)
]

result = differential_evolution(
    objective_function, 
    bounds, 
    strategy='best1bin', 
    maxiter=15,      # Количество поколений (увеличьте до 20-50 для точности)
    popsize=10,       # Размер популяции (увеличьте до 10-15)
    disp=True,
    polish=True,
    workers=1
)

print("Оптимальные параметры найдены:")
print(result.x)

Simulating: [2.04845814e+02 9.79412231e+04 2.18497645e+03 4.74527910e+04
 4.92267951e+02 6.85127814e+03 7.63777786e+01]
Error: 14757.08
Simulating: [2.98898073e+02 4.34043893e+05 2.64682207e+03 1.88577916e+04
 4.23935248e+02 3.74372523e+03 1.29148236e+01]
Error: 8497.43
Simulating: [2.91618801e+02 4.40158151e+05 2.47085227e+03 4.54382228e+04
 2.12239316e+02 6.68293260e+03 6.09223060e+01]
Error: 70209.05
Simulating: [3.38676158e+02 2.58251964e+05 1.01627538e+02 2.36229106e+03
 3.51658314e+02 1.56979844e+03 2.09818783e+01]
Error: 1979227.41
Simulating: [3.07035947e+02 4.52067811e+05 3.28115388e+03 2.78142868e+04
 2.78475998e+02 8.01883618e+03 5.35346991e+01]
Error: 21385.23
Simulating: [3.66227829e+02 3.64399535e+05 8.69113630e+02 3.05986512e+04
 1.61419641e+02 3.82512481e+03 4.75424907e+01]
Error: 324913.77
Simulating: [3.34051832e+02 1.12603581e+05 3.98346318e+03 3.94831652e+04
 2.40571683e+02 3.30510004e+03 7.27121790e+01]
Error: 3933.51
Simulating: [2.41295583e+02 4.07993819e+05 4.71